In [ ]:
import os, sys, json, csv, math, time, subprocess, warnings, re
from pathlib import Path

import numpy as np, pandas as pd, torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, confusion_matrix

import warnings
warnings.filterwarnings(
    "ignore",
    message="enable_nested_tensor is True",
    module="torch.nn.modules.transformer"
)

# ---------------- Repro/Device ----------------
# Device priority: CUDA (NVIDIA) -> MPS (Apple Silicon) -> CPU.
PREFERRED_GPU_INDEX = 1
if torch.cuda.is_available():
    selected_gpu_index = min(PREFERRED_GPU_INDEX, torch.cuda.device_count() - 1)
    torch.cuda.set_device(selected_gpu_index)
    active_device = torch.device(f"cuda:{selected_gpu_index}")
    selected_accelerator = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    selected_gpu_index = None
    active_device = torch.device("mps")
    selected_accelerator = "mps"
else:
    selected_gpu_index = None
    active_device = torch.device("cpu")
    selected_accelerator = "cpu"

print(f"Torch {torch.__version__} | CUDA: {torch.cuda.is_available()} | Device: {active_device}")
if selected_accelerator == "cuda":
    print("Current GPU:", torch.cuda.get_device_name(torch.cuda.current_device()))
elif selected_accelerator == "mps":
    print("Using Apple Silicon MPS accelerator.")
else:
    print("Using CPU backend.")

# Compatibility workaround: some Torch builds fail while importing torch._dynamo
# via @torch._disable_dynamo wrappers used in optimizer methods.
def patch_optimizer_dynamo_wrappers():
    optimizer_class = torch.optim.Optimizer
    method_names = ["add_param_group", "zero_grad", "state_dict", "load_state_dict"]
    for method_name in method_names:
        wrapped_method = getattr(optimizer_class, method_name, None)
        raw_method = getattr(wrapped_method, "__wrapped__", None)
        if raw_method is not None and not hasattr(raw_method, "__dynamo_disable"):
            raw_method.__dynamo_disable = raw_method

patch_optimizer_dynamo_wrappers()

# Additional compatibility patch: unwrap Adam/AdamW step to avoid `_use_grad`
# importing torch._dynamo at runtime on affected builds.
def patch_adam_step_wrappers():
    for optimizer_class in [torch.optim.Adam, torch.optim.AdamW]:
        step_fn = getattr(optimizer_class, "step", None)
        while hasattr(step_fn, "__wrapped__"):
            step_fn = step_fn.__wrapped__
        optimizer_class.step = step_fn
        # Prevent Optimizer._patch_step_function from wrapping this again.
        optimizer_class.step.hooked = True

patch_adam_step_wrappers()

# Install optional runtime dependencies if missing.
for package_name in ("optuna", "tqdm", "matplotlib", "spikingjelly"):
    try:
        __import__(package_name)
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])
import optuna
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import torch.nn.functional as F
import types

# --- SNN imports (SpikingJelly) ---
# Compatibility workaround: some torch/spikingjelly builds fail when TorchScript
# compiles surrogate SoftSign during import.
_original_torch_jit_script = torch.jit.script

def _safe_torch_jit_script(obj=None, *args, **kwargs):
    if obj is None:
        def _decorator(fn):
            return _safe_torch_jit_script(fn, *args, **kwargs)
        return _decorator
    try:
        return _original_torch_jit_script(obj, *args, **kwargs)
    except Exception as error:
        error_text = str(error)
        if "softsign" in error_text and "Argument lower not provided" in error_text:
            return obj
        raise

torch.jit.script = _safe_torch_jit_script
try:
    from spikingjelly.activation_based import neuron, surrogate, functional
except Exception as import_error:
    print(f"Warning: SpikingJelly import failed ({import_error}). Using local LIF fallback.")

    class _FallbackLIFNode(nn.Module):
        def __init__(self, tau=2.0, v_threshold=0.2, v_reset=0.0, surrogate_function=None, detach_reset=True):
            super().__init__()
            self.tau = float(tau)
            self.v_threshold = float(v_threshold)
            self.v_reset = float(v_reset)
            self.v = None

        def reset(self):
            self.v = None

        def forward(self, x):
            if self.v is None or self.v.shape != x.shape:
                self.v = torch.zeros_like(x)
            self.v = self.v + (x - self.v) / self.tau
            spikes = (self.v >= self.v_threshold).to(x.dtype)
            self.v = torch.where(spikes > 0, torch.full_like(self.v, self.v_reset), self.v)
            return spikes

    neuron = types.SimpleNamespace(LIFNode=_FallbackLIFNode)
    surrogate = types.SimpleNamespace(ATan=lambda: None)
    functional = types.SimpleNamespace(
        reset_net=lambda module: module.reset() if hasattr(module, "reset") else None
    )
finally:
    torch.jit.script = _original_torch_jit_script

# ---------------- Config ----------------
DATA_CSV_PATH = Path("Base.csv")
RANDOM_SEED = 42
WEIGHT_DECAY = 1e-5
FPR_CAP = 0.05

# Search settings
N_TRIALS = 100
TRIAL_EPOCHS = 8
FINAL_EPOCHS = 20

# CSV logs (unchanged format)
TRIALS_CSV = Path("snn_ftt_100_trials_Base.csv")
BEST_CSV = Path("snn_ftt_100_trials_best_Base.csv")
TRIAL_LOG_FIELDS = [
    "trial", "epoch", "d_token", "n_blocks", "n_heads", "ffn_hidden", "dropout", "lr", "batch",
    "val_auc", "val_prauc", "val_recall_at_fpr", "val_fpr", "val_threshold"
]

# Per-trial fairness/performance dump used for plotting.
FAIR_CSV = Path("snn_ftt_100_trials_fairness_Base.csv")
FAIR_LOG_FIELDS = [
    "trial", "perf_recall_test", "fairness_ratio_test", "thr_star", "fpr_test",
    "recall_val", "fpr_val", "fairness_ratio_val"
]

# Export directory for final model + plots.
EXPORT_DIR = Path("snn_ftt_100_export_Base")

# ---------------- Repro/Device ----------------
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print(f"Torch {torch.__version__} | CUDA: {torch.cuda.is_available()} | Device: {active_device}")
if active_device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(torch.cuda.current_device()))
elif active_device.type == "mps":
    print("MPS backend active.")
else:
    print("CPU backend active.")

# ---------------- Data prep ----------------
def resolve_dataset_path(raw_path: Path) -> Path:
    # Allow both "Base.csv" and "baf-datasets/Base.csv" style paths.
    candidate_paths = [
        raw_path,
        Path("baf-datasets") / raw_path.name,
    ]
    for candidate_path in candidate_paths:
        if candidate_path.exists():
            return candidate_path
    raise FileNotFoundError(
        f"Dataset not found. Tried: {', '.join(str(path) for path in candidate_paths)}"
    )


DATA_CSV_PATH = resolve_dataset_path(DATA_CSV_PATH)
full_df = pd.read_csv(DATA_CSV_PATH)

# Detect the target column from common names.
for candidate_target_col in ["fraud_bool", "fraud", "is_fraud", "label", "target"]:
    if candidate_target_col in full_df.columns:
        target_col = candidate_target_col
        break
else:
    raise ValueError("Set target column name.")

# Time split by month: 0-5 for train/valid, >=6 for test.
if "month" not in full_df.columns:
    raise ValueError("Expected a 'month' column for time-based split.")


def parse_int_from_value(value):
    try:
        return int(re.findall(r"-?\d+", str(value))[0])
    except Exception:
        return np.nan


month_values = full_df["month"].map(parse_int_from_value)
if month_values.isna().any():
    raise ValueError("Could not parse some 'month' values as integers.")
full_df = full_df.assign(month=month_values.astype(int))

train_valid_df = full_df[full_df["month"] <= 5].copy()
test_df = full_df[full_df["month"] >= 6].copy()
if len(train_valid_df) == 0 or len(test_df) == 0:
    raise ValueError("Time split produced empty train/valid or test set. Check 'month' range in CSV.")

# Build feature lists.
ignored_name_fragments = {"customer", "id", "uuid"}
feature_cols = [
    column_name
    for column_name in full_df.columns
    if column_name != target_col
    and column_name != "month"
    and not any(fragment in column_name.lower() for fragment in ignored_name_fragments)
]

categorical_cols = [
    column_name for column_name in feature_cols
    if str(full_df[column_name].dtype) in ("object", "category")
]
for column_name in feature_cols:
    if (
        column_name not in categorical_cols
        and pd.api.types.is_integer_dtype(full_df[column_name])
        and full_df[column_name].nunique() <= 50
    ):
        categorical_cols.append(column_name)

continuous_cols = [
    column_name
    for column_name in feature_cols
    if column_name not in categorical_cols and pd.api.types.is_numeric_dtype(full_df[column_name])
]

# Fairness groups: age>=50 vs <50.
def extract_numeric_age(series: pd.Series):
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce")
    numeric_text = series.astype(str).str.extract(r"(-?\d+)", expand=False)
    return pd.to_numeric(numeric_text, errors="coerce")


def make_age_group_mask(dataframe: pd.DataFrame):
    for age_column_name in ["age", "Age", "AGE", "age_group"]:
        if age_column_name in dataframe.columns:
            parsed_age = extract_numeric_age(dataframe[age_column_name])
            group_mask = (parsed_age >= 50).astype(int)
            return group_mask.fillna(0).values.astype(int)
    return np.zeros(len(dataframe), dtype=int)


# Split train/valid inside months 0-5.
from sklearn.model_selection import train_test_split as sklearn_train_test_split

train_df, valid_df = sklearn_train_test_split(
    train_valid_df,
    test_size=0.2,
    stratify=train_valid_df[target_col],
    random_state=RANDOM_SEED,
)

train_group_mask = make_age_group_mask(train_df)
valid_group_mask = make_age_group_mask(valid_df)
test_group_mask = make_age_group_mask(test_df)

# Lock categorical vocab on TRAIN(0-5) and reserve one UNK index per feature.
categorical_levels = {}
categorical_cardinalities = {}
for column_name in categorical_cols:
    known_levels = train_valid_df[column_name].astype("category").cat.categories
    categorical_levels[column_name] = list(known_levels)
    categorical_cardinalities[column_name] = len(known_levels) + 1  # +1 for UNK

cat_cardinalities = [categorical_cardinalities[column_name] for column_name in categorical_cols] if categorical_cols else None

# Train means for continuous features (used for imputation everywhere).
continuous_train_mean = train_valid_df[continuous_cols].astype(float).mean() if continuous_cols else None


def encode_categorical_features(dataframe_part: pd.DataFrame) -> torch.Tensor | None:
    if not categorical_cols:
        return None
    encoded_columns = []
    for column_name in categorical_cols:
        raw_codes = pd.Categorical(dataframe_part[column_name], categories=categorical_levels[column_name]).codes
        unknown_index = categorical_cardinalities[column_name] - 1
        safe_codes = np.where(raw_codes == -1, unknown_index, raw_codes).astype("int64")
        encoded_columns.append(safe_codes)
    stacked = np.stack(encoded_columns, axis=1) if encoded_columns else None
    return torch.as_tensor(stacked, dtype=torch.long) if stacked is not None else None


def encode_continuous_features(dataframe_part: pd.DataFrame) -> torch.Tensor | None:
    if not continuous_cols:
        return None
    numeric_values = dataframe_part[continuous_cols].astype(float).copy()
    numeric_values = numeric_values.fillna(continuous_train_mean)
    return torch.as_tensor(numeric_values.values, dtype=torch.float32)


def build_split_tensors(dataframe_part: pd.DataFrame):
    x_num = encode_continuous_features(dataframe_part)
    x_cat = encode_categorical_features(dataframe_part)
    y_target = torch.as_tensor(dataframe_part[target_col].values, dtype=torch.float32).view(-1, 1)
    return x_num, x_cat, y_target


# Build tensors for each split.
x_num_train, x_cat_train, y_train = build_split_tensors(train_df)
x_num_valid, x_cat_valid, y_valid = build_split_tensors(valid_df)
x_num_test, x_cat_test, y_test = build_split_tensors(test_df)

# Normalize continuous features using TRAIN stats only.
if x_num_train is not None:
    train_mean = x_num_train.mean(0, keepdim=True)
    train_std = x_num_train.std(0, keepdim=True).clamp_min(1e-6)
    x_num_train = (x_num_train - train_mean) / train_std
    if x_num_valid is not None:
        x_num_valid = (x_num_valid - train_mean) / train_std
    if x_num_test is not None:
        x_num_test = (x_num_test - train_mean) / train_std


def move_batch_tensors_to_device(x_num, x_cat, y_target):
    batch_parts = []
    if x_num is not None:
        batch_parts.append(x_num.to(active_device))
    if x_cat is not None:
        batch_parts.append(x_cat.to(active_device))
    batch_parts.append(y_target.to(active_device))
    return batch_parts


# Number of numeric features used by tokenizer.
num_numeric_features = 0 if x_num_train is None else x_num_train.shape[1]

# Sanity check: ensure categorical codes never exceed embedding cardinalities.
if x_cat_train is not None:
    for column_index, column_name in enumerate(categorical_cols):
        max_seen_code = int(max(
            (int(x_cat_train[:, column_index].max().item()) if x_cat_train.numel() else 0),
            (int(x_cat_valid[:, column_index].max().item()) if x_cat_valid is not None and x_cat_valid.numel() else 0),
            (int(x_cat_test[:, column_index].max().item()) if x_cat_test is not None and x_cat_test.numel() else 0),
        ))
        assert max_seen_code < categorical_cardinalities[column_name], (
            f"Index overflow in '{column_name}': {max_seen_code} >= {categorical_cardinalities[column_name]}"
        )


# ---------------- FT-Transformer with SNN Head ----------------
class FeatureTokenizer(nn.Module):
    def __init__(self, n_num_features: int, cat_feature_cardinalities, d_token: int):
        super().__init__()
        self.n_num_features = n_num_features
        self.cat_feature_cardinalities = cat_feature_cardinalities or []
        self.d_token = d_token

        if self.n_num_features > 0:
            self.num_weight = nn.Parameter(torch.randn(self.n_num_features, d_token) * 0.02)
            self.num_bias = nn.Parameter(torch.zeros(self.n_num_features, d_token))
        else:
            self.register_parameter("num_weight", None)
            self.register_parameter("num_bias", None)

        self.cat_embeds = nn.ModuleList([
            nn.Embedding(cardinality, d_token)
            for cardinality in self.cat_feature_cardinalities
        ])
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_token) * 0.02)

    def forward(self, x_num, x_cat):
        batch_size = x_num.size(0) if x_num is not None else x_cat.size(0)
        token_blocks = []

        if x_num is not None:
            token_blocks.append(
                x_num.unsqueeze(-1) * self.num_weight.unsqueeze(0) + self.num_bias.unsqueeze(0)
            )

        if x_cat is not None and len(self.cat_feature_cardinalities) > 0:
            cat_embeddings = [embedding(x_cat[:, idx]) for idx, embedding in enumerate(self.cat_embeds)]
            token_blocks.append(torch.stack(cat_embeddings, dim=1))

        if not token_blocks:
            raise ValueError("No features to tokenize")

        all_tokens = torch.cat(token_blocks, dim=1)
        all_tokens = torch.cat([self.cls_token.expand(batch_size, 1, -1), all_tokens], dim=1)
        return all_tokens


class SNNHead(nn.Module):
    """
    Converts CLS token [B, d] to one fraud logit [B, 1] using LIF dynamics.
    A static CLS vector is injected for T steps, membrane states are averaged,
    and a linear readout produces the final logit.
    """

    def __init__(self, d_token, T=8, tau=2.0, v_th=0.2, v_reset=0.0):
        super().__init__()
        self.T = T
        self.batch_norm = nn.BatchNorm1d(d_token, affine=True)
        self.input_proj = nn.Linear(d_token, d_token)
        nn.init.kaiming_normal_(self.input_proj.weight, nonlinearity="linear")
        nn.init.zeros_(self.input_proj.bias)

        self.lif = neuron.LIFNode(
            tau=tau,
            v_threshold=v_th,
            v_reset=v_reset,
            surrogate_function=surrogate.ATan(),
            detach_reset=True,
        )
        self.log_alpha = nn.Parameter(torch.zeros(1))
        self.current_bias = nn.Parameter(torch.zeros(d_token))
        self.readout = nn.Linear(d_token, 1)
        nn.init.zeros_(self.readout.bias)

    def forward(self, cls_embedding):
        # Reset neuron state between mini-batches.
        try:
            functional.reset_net(self.lif)
        except Exception:
            if hasattr(self.lif, "reset"):
                self.lif.reset()

        normalized = self.batch_norm(cls_embedding)
        projected = self.input_proj(normalized)
        gain = F.softplus(self.log_alpha) + 1.0
        injected_current = projected * gain + self.current_bias

        membrane_sum = 0.0
        for _ in range(self.T):
            _ = self.lif(injected_current)
            membrane_sum = membrane_sum + self.lif.v

        membrane_avg = membrane_sum / self.T
        return self.readout(membrane_avg)


class FTTransformer(nn.Module):
    def __init__(
        self,
        n_num_features,
        cat_feature_cardinalities,
        d_token=32,
        n_blocks=4,
        n_heads=8,
        ffn_hidden=128,
        dropout=0.2,
    ):
        super().__init__()
        self.tokenizer = FeatureTokenizer(n_num_features, cat_feature_cardinalities, d_token)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_token,
            nhead=n_heads,
            dim_feedforward=ffn_hidden,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_blocks)
        self.head = SNNHead(d_token=d_token, T=8)

    def forward(self, x_num, x_cat):
        token_sequence = self.tokenizer(x_num, x_cat)
        encoded_tokens = self.encoder(token_sequence)
        cls_embedding = encoded_tokens[:, 0, :]
        return self.head(cls_embedding)


# ---------------- Metrics helpers ----------------
def recall_at_fpr_cap(y_true_tensor, y_probabilities, cap=FPR_CAP):
    y_true_np = y_true_tensor.cpu().numpy().ravel()
    fpr_values, tpr_values, thresholds = roc_curve(y_true_np, y_probabilities)
    under_cap_mask = fpr_values <= cap
    if not np.any(under_cap_mask):
        return 0.0, 0.5, 1.0

    best_index_within_cap = np.argmax(tpr_values[under_cap_mask])
    return (
        tpr_values[under_cap_mask][best_index_within_cap],
        thresholds[under_cap_mask][best_index_within_cap],
        fpr_values[under_cap_mask][best_index_within_cap],
    )
# False Positive Rate (FPR) = FP / (FP + TN): among actual negatives, how many were wrongly predicted positive.
# Fairness ratio at this threshold: max(group FPR) / min(group FPR); 1.0 is ideal (equal FPRs across groups).

def fairness_ratio_at_threshold(y_true, y_probabilities, group_mask, threshold):
    """Predictive equality metric: ratio of group FPRs (ideal is 1)."""
    y_true_int = y_true.ravel().astype(int)
    y_pred_int = (y_probabilities >= threshold).astype(int)

    group_fprs = []
    for group_id in [0, 1]:
        group_rows = group_mask == group_id
        group_negatives = group_rows & (y_true_int == 0)
        if group_negatives.sum() == 0:
            group_fprs.append(0.0)
            continue

        false_positives = np.logical_and(y_pred_int == 1, y_true_int == 0)[group_rows].sum()
        true_negatives = np.logical_and(y_pred_int == 0, y_true_int == 0)[group_rows].sum()
        # Add a tiny epsilon (1e-12) to avoid division-by-zero when a group has no negative samples.
        group_fpr = false_positives / (false_positives + true_negatives + 1e-12)
        group_fprs.append(float(group_fpr))

    low_fpr, high_fpr = min(group_fprs), max(group_fprs)
    return high_fpr / (low_fpr + 1e-12), group_fprs


# ---------------- Objective ----------------
# Start fairness CSV fresh for this run.
FAIR_CSV.unlink(missing_ok=True)


def build_loader(x_num, x_cat, y_target, batch_size, shuffle=False):
    return DataLoader(
        TensorDataset(*move_batch_tensors_to_device(x_num, x_cat, y_target)),
        batch_size=batch_size,
        shuffle=shuffle,
    )


def objective(trial: optuna.Trial):
    d_token = trial.suggest_categorical("d_token", [16, 32, 48, 64])
    n_blocks = trial.suggest_int("n_blocks", 2, 6)
    n_heads = trial.suggest_categorical("n_heads", [4, 8])
    ffn_hidden = trial.suggest_categorical("ffn_hidden", [64, 128, 192, 256])
    dropout = trial.suggest_float("dropout", 0.0, 0.4)
    learning_rate = trial.suggest_float("lr", 3e-4, 3e-3, log=True)
    batch_size = trial.suggest_categorical("batch", [1024, 2048, 4096, 8192])

    model = FTTransformer(
        num_numeric_features,
        cat_cardinalities,
        d_token,
        n_blocks,
        n_heads,
        ffn_hidden,
        dropout,
    ).to(active_device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.BCEWithLogitsLoss()

    train_loader = build_loader(x_num_train, x_cat_train, y_train, batch_size, shuffle=True)
    valid_loader = build_loader(x_num_valid, x_cat_valid, y_valid, batch_size * 2, shuffle=False)
    test_loader = build_loader(x_num_test, x_cat_test, y_test, batch_size * 2, shuffle=False)

    def predict_probabilities(data_loader):
        model.eval()
        probability_chunks = []
        with torch.no_grad():
            for *features, _ in data_loader:
                probability_chunks.append(torch.sigmoid(model(*features)).squeeze(1).cpu())
        return torch.cat(probability_chunks).numpy()

    best_valid_recall = -1.0
    best_trial_row = None
    best_model_state = None
    best_validation_threshold = 0.5

    for epoch_idx in tqdm(range(1, TRIAL_EPOCHS + 1), leave=False, desc=f"Trial {trial.number}"):
        model.train()
        for *features, batch_targets in train_loader:
            logits = model(*features)
            loss = loss_fn(logits, batch_targets)
            optimizer.zero_grad()
            loss.backward()
            with torch.no_grad():
                optimizer.step()

        valid_probs = predict_probabilities(valid_loader)
        valid_auc = roc_auc_score(y_valid.cpu().numpy(), valid_probs)
        valid_prauc = average_precision_score(y_valid.cpu().numpy(), valid_probs)
        valid_recall, valid_threshold, valid_fpr = recall_at_fpr_cap(y_valid, valid_probs, FPR_CAP)

        trial_log_row = {
            "trial": trial.number,
            "epoch": epoch_idx,
            "d_token": d_token,
            "n_blocks": n_blocks,
            "n_heads": n_heads,
            "ffn_hidden": ffn_hidden,
            "dropout": dropout,
            "lr": learning_rate,
            "batch": batch_size,
            "val_auc": valid_auc,
            "val_prauc": valid_prauc,
            "val_recall_at_fpr": valid_recall,
            "val_fpr": valid_fpr,
            "val_threshold": valid_threshold,
        }

        write_header = not TRIALS_CSV.exists()
        with open(TRIALS_CSV, "a", newline="") as csv_file:
            writer = csv.DictWriter(csv_file, fieldnames=TRIAL_LOG_FIELDS)
            if write_header:
                writer.writeheader()
            writer.writerow(trial_log_row)

        if valid_recall > best_valid_recall:
            best_valid_recall = valid_recall
            best_trial_row = trial_log_row
            best_model_state = {
                name: tensor.detach().cpu().clone()
                for name, tensor in model.state_dict().items()
            }
            best_validation_threshold = float(valid_threshold)

        trial.report(best_valid_recall, epoch_idx)
        if trial.should_prune():
            break

    # Log the best row for this trial.
    write_header = not BEST_CSV.exists()
    with open(BEST_CSV, "a", newline="") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=TRIAL_LOG_FIELDS)
        if write_header:
            writer.writeheader()
        writer.writerow(best_trial_row)

    # Fairness/performance snapshot (validation + test) at best state/threshold.
    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    valid_probs = predict_probabilities(valid_loader)
    recall_valid, _, fpr_valid = recall_at_fpr_cap(y_valid, valid_probs, FPR_CAP)

    threshold_star = best_validation_threshold
    test_probs = predict_probabilities(test_loader)
    y_test_np = y_test.cpu().numpy().ravel()
    y_test_pred = (test_probs >= threshold_star).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test_np, y_test_pred).ravel()
    recall_test = tp / (tp + fn + 1e-12)
    fpr_test = fp / (fp + tn + 1e-12)

    fairness_ratio_valid, _ = fairness_ratio_at_threshold(
        y_valid.cpu().numpy().ravel(), valid_probs, valid_group_mask, threshold_star
    )
    fairness_ratio_test, _ = fairness_ratio_at_threshold(
        y_test_np, test_probs, test_group_mask, threshold_star
    )

    write_header = not FAIR_CSV.exists()
    with open(FAIR_CSV, "a", newline="") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=FAIR_LOG_FIELDS)
        if write_header:
            writer.writeheader()
        writer.writerow({
            "trial": trial.number,
            "perf_recall_test": recall_test,
            "fairness_ratio_test": fairness_ratio_test,
            "thr_star": threshold_star,
            "fpr_test": fpr_test,
            "recall_val": recall_valid,
            "fpr_val": fpr_valid,
            "fairness_ratio_val": fairness_ratio_valid,
        })

    return best_valid_recall


# ---------------- Run study ----------------
study = optuna.create_study(direction="maximize", study_name="snn_ftt_recall_at_fpr_cap")
print("Running hyperparameter search...")
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
print("Best recall@FPR<=5%:", study.best_value)
print("Best params:", study.best_params)


# ---------------- Retrain best config ----------------
best_params = study.best_params
model = FTTransformer(
    num_numeric_features,
    cat_cardinalities,
    d_token=best_params["d_token"],
    n_blocks=best_params["n_blocks"],
    n_heads=best_params["n_heads"],
    ffn_hidden=best_params["ffn_hidden"],
    dropout=best_params["dropout"],
).to(active_device)
optimizer = torch.optim.AdamW(model.parameters(), lr=best_params["lr"], weight_decay=WEIGHT_DECAY)
loss_fn = nn.BCEWithLogitsLoss()

train_loader = build_loader(x_num_train, x_cat_train, y_train, best_params["batch"], shuffle=True)
valid_loader = build_loader(x_num_valid, x_cat_valid, y_valid, best_params["batch"] * 2, shuffle=False)
test_loader = build_loader(x_num_test, x_cat_test, y_test, best_params["batch"] * 2, shuffle=False)


def predict_probabilities(data_loader):
    model.eval()
    probability_chunks = []
    with torch.no_grad():
        for *features, _ in data_loader:
            probability_chunks.append(torch.sigmoid(model(*features)).squeeze(1).cpu())
    return torch.cat(probability_chunks).numpy()


best_valid_recall = -1.0
best_model_state = None
for epoch_idx in range(1, FINAL_EPOCHS + 1):
    model.train()
    for *features, batch_targets in train_loader:
        logits = model(*features)
        loss = loss_fn(logits, batch_targets)
        optimizer.zero_grad()
        loss.backward()
        with torch.no_grad():
            optimizer.step()

    valid_probs = predict_probabilities(valid_loader)
    valid_recall, threshold_star, _ = recall_at_fpr_cap(y_valid, valid_probs, FPR_CAP)
    if valid_recall > best_valid_recall:
        best_valid_recall = valid_recall
        best_model_state = {
            name: tensor.detach().cpu().clone()
            for name, tensor in model.state_dict().items()
        }

    if epoch_idx % 2 == 0:
        print(f"[Retrain] Epoch {epoch_idx:02d} | VALID recall@FPR<=5% = {valid_recall:.4f}")

model.load_state_dict(best_model_state)


# ---------------- Final evaluation ----------------
valid_probs = predict_probabilities(valid_loader)
test_probs = predict_probabilities(test_loader)
print("\nVALID AUC:", roc_auc_score(y_valid.cpu().numpy(), valid_probs))
print("VALID PR-AUC:", average_precision_score(y_valid.cpu().numpy(), valid_probs))
valid_recall, threshold_star, valid_fpr = recall_at_fpr_cap(y_valid, valid_probs, FPR_CAP)
print(f"VALID recall@FPR<={int(FPR_CAP * 100)}%: {valid_recall:.4f} | chosen_thr: {threshold_star:.6f} | FPR: {valid_fpr:.4f}")

y_test_np = y_test.cpu().numpy().ravel()
y_test_pred = (test_probs >= threshold_star).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test_np, y_test_pred).ravel()
recall_test = tp / (tp + fn + 1e-12)
fpr_test = fp / (fp + tn + 1e-12)
print(f"TEST  recall: {recall_test:.4f} | TEST FPR: {fpr_test:.4f}")
print("TEST  AUC:", roc_auc_score(y_test.cpu().numpy(), test_probs))
print("TEST  PR-AUC:", average_precision_score(y_test.cpu().numpy(), test_probs))


# ---------------- Save artifacts ----------------
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), EXPORT_DIR / "model.pt")
with open(EXPORT_DIR / "meta.json", "w") as meta_file:
    json.dump({
        "target": target_col,
        "cat_cols": categorical_cols,
        "cont_cols": continuous_cols,
        "fpr_cap": FPR_CAP,
        "threshold": float(threshold_star),
        **study.best_params,
    }, meta_file, indent=2)

print(f"\nSaved model + meta to {EXPORT_DIR}/")
print(f"Per-epoch trial log: {TRIALS_CSV}")
print(f"Best-per-trial log : {BEST_CSV}")
print(f"Fairness log       : {FAIR_CSV}")


# ---------------- Plots (fairness vs performance) ----------------
fairness_results_df = pd.read_csv(FAIR_CSV)

# 1) Full scatter of all trials.
fig, ax = plt.subplots(figsize=(7.0, 5.0), dpi=150)
ax.scatter(fairness_results_df["perf_recall_test"], fairness_results_df["fairness_ratio_test"], alpha=0.7)

top_5_by_recall = fairness_results_df.sort_values("perf_recall_test", ascending=False).head(5)
ax.scatter(
    top_5_by_recall["perf_recall_test"],
    top_5_by_recall["fairness_ratio_test"],
    s=70,
    edgecolors="black",
)
ax.set_xlabel("Performance (Recall @ 5% FPR) - TEST")
ax.set_ylabel("Fairness (ratio of group FPRs) - TEST (ideal = 1)")
ax.set_title("Fairness vs Performance (all trials)")
plt.tight_layout()
plt.savefig(EXPORT_DIR / "fig_fairness_vs_performance.png", bbox_inches="tight")
print("Saved:", EXPORT_DIR / "fig_fairness_vs_performance.png")

# 2) Zoom into high-performance region.
zoom_xmin = max(
    fairness_results_df["perf_recall_test"].min(),
    fairness_results_df["perf_recall_test"].quantile(0.6),
)
zoom_xmax = fairness_results_df["perf_recall_test"].max()
zoom_ymin = fairness_results_df["fairness_ratio_test"].quantile(0.05)
zoom_ymax = fairness_results_df["fairness_ratio_test"].quantile(0.95)

fig, ax = plt.subplots(figsize=(7.0, 5.0), dpi=150)
ax.scatter(fairness_results_df["perf_recall_test"], fairness_results_df["fairness_ratio_test"], alpha=0.5)
ax.scatter(
    top_5_by_recall["perf_recall_test"],
    top_5_by_recall["fairness_ratio_test"],
    s=70,
    edgecolors="black",
)
ax.set_xlim(zoom_xmin, zoom_xmax + 1e-6)
ax.set_ylim(zoom_ymin - 1e-3, zoom_ymax + 1e-3)
ax.set_xlabel("Performance (Recall @ 5% FPR) - TEST")
ax.set_ylabel("Fairness (ratio of group FPRs) - TEST (ideal = 1)")
ax.set_title("Fairness vs Performance (zoom)")
plt.tight_layout()
plt.savefig(EXPORT_DIR / "fig_fairness_vs_performance_zoom.png", bbox_inches="tight")
print("Saved:", EXPORT_DIR / "fig_fairness_vs_performance_zoom.png")


Torch 2.8.0 | CUDA: False | Device: mps
Using Apple Silicon MPS accelerator.
Torch 2.8.0 | CUDA: False | Device: mps
MPS backend active.


[I 2026-02-08 13:26:19,535] A new study created in memory with name: snn_ftt_recall_at_fpr_cap


Running hyperparameter search...


Best trial: 0. Best value: 0.531288:   1%|          | 1/100 [09:16<15:17:39, 556.16s/it]

[I 2026-02-08 13:35:35,693] Trial 0 finished with value: 0.5312883435582823 and parameters: {'d_token': 64, 'n_blocks': 5, 'n_heads': 4, 'ffn_hidden': 64, 'dropout': 0.08080484821768397, 'lr': 0.0014725077940151811, 'batch': 4096}. Best is trial 0 with value: 0.5312883435582823.


...

[I 2026-02-08 17:32:11,444] Trial 73 finished with value: 0.5349693251533743 and parameters: {'d_token': 48, 'n_blocks': 2, 'n_heads': 8, 'ffn_hidden': 64, 'dropout': 0.3000792914048591, 'lr': 0.0018406103280913845, 'batch': 1024}. Best is trial 54 with value: 0.5392638036809816.


Best trial: 54. Best value: 0.539264:  75%|███████▌  | 75/100 [4:08:02<1:21:17, 195.09s/it]

[I 2026-02-08 17:34:22,108] Trial 74 finished with value: 0.5153374233128835 and parameters: {'d_token': 48, 'n_blocks': 2, 'n_heads': 8, 'ffn_hidden': 64, 'dropout': 0.3279911885970953, 'lr': 0.0017922868680047403, 'batch': 1024}. Best is trial 54 with value: 0.5392638036809816.


Best trial: 54. Best value: 0.539264:  76%|███████▌  | 76/100 [4:15:12<1:46:15, 265.65s/it]

[I 2026-02-08 17:41:32,392] Trial 75 finished with value: 0.5269938650306748 and parameters: {'d_token': 48, 'n_blocks': 5, 'n_heads': 8, 'ffn_hidden': 64, 'dropout': 0.04748753803195116, 'lr': 0.0024486142819198312, 'batch': 1024}. Best is trial 54 with value: 0.5392638036809816.
